# 06 — Energy

Why Palau sits where it does. Notebook 03 puts Palau at the top of the Pacific
ASR; this notebook supplies the mechanism the piece can defend, from official
data: **Palau burns more oil-fired electricity per resident than any other
Pacific country.**

Two measures, because they answer different questions:

| Measure | Question | Pacific | World |
|---|---|---|---|
| electricity mix by source | what is the power made from? | PDH `DF_POWER_GEN`, 18 islands, 2000–2023 | OWID energy, ~210 countries |
| renewable share of total final energy (SDG 7.2.1) | how much of *all* energy — power, transport, cooking — is renewable? | PDH `DF_SDG` `EG_FEC_RNEW`, 20 islands, 2000–2022 | World Bank `EG.FEC.RNEW.ZS` |

**"Oil", not "diesel".** Every dataset's category is petroleum products as a
class. Pacific island power stations run on diesel, so the description is fair,
but no source labels it that way — write "oil-fired", not "diesel share", when
citing a number.

**Outputs**
- `data_viz/energy.csv` — Pacific island-years 2000–2023, plus every other
  country at the snapshot year
- `data_viz/energy.json` — snapshot with world ranks, world medians, and the
  full Pacific series

In [ ]:
import json

import pandas as pd
import pycountry
import requests

from config import VIZ, YEARS
from pdh_api import fetch_data_pacific, fetch_structure_pacific

START, END = str(min(YEARS)), str(max(YEARS))
YEAR_SNAP = max(YEARS)  # 2023

AGGREGATES = {"_T", "_TXPNG", "MEL", "MELXPNG", "MIC", "POL"}

# The generation sources DF_POWER_GEN actually carries for the Pacific. The
# dataflow defines 21; the other 13 (nuclear, offshore wind, marine, pumped
# storage...) have no Pacific rows.
SOURCES = ["OIL", "COAL", "GAS", "SOLPV", "HYDRO", "WNDON", "BIOSLD", "GEOTH"]
FOSSIL = ["OIL", "COAL", "GAS"]

## 1. Pacific — generation by source

`DF_POWER_GEN` reports GWh generated per source per year. Summing the fossil
three against the renewable five gives the mix; there is no need for the
dataflow's own `RENTOT` / `NRENTOT` totals, and skipping them avoids
double-counting when both are present.

In [ ]:
gen = fetch_data_pacific(
    source="DF_POWER_GEN",
    start_period=START,
    end_period=END,
    key="A.." + "+".join(SOURCES) + "._T",  # _T = on-grid and off-grid
)

gen = (
    gen[~gen["GEO_PICT"].isin(AGGREGATES)]
    .assign(year=lambda d: d["TIME_PERIOD"].astype(int))
    .pivot_table(index=["GEO_PICT", "year"], columns="ENERGY_SOURCE",
                 values="value")
    .reset_index()
)
for src in SOURCES:
    if src not in gen:
        gen[src] = 0.0
gen[SOURCES] = gen[SOURCES].fillna(0.0)

gen["total_gwh"] = gen[SOURCES].sum(axis=1)
gen = gen[gen["total_gwh"] > 0]
gen["oil_share_elec"] = gen["OIL"] / gen["total_gwh"] * 100
gen["fossil_share_elec"] = gen[FOSSIL].sum(axis=1) / gen["total_gwh"] * 100
gen["renewables_share_elec"] = 100 - gen["fossil_share_elec"]

print(f"{gen['GEO_PICT'].nunique()} islands, {gen['year'].min()}-{gen['year'].max()}, "
      f"{len(gen)} island-years")

In [ ]:
pop = fetch_data_pacific(
    source="DF_POP_PROJ",
    start_period=START,
    end_period=END,
    key="A..MIDYEARPOPEST._T._T",
    v="3.0",
)
pop = (
    pop[~pop["GEO_PICT"].isin(AGGREGATES)]
    .assign(year=lambda d: d["TIME_PERIOD"].astype(int))
    .rename(columns={"value": "population"})
    [["GEO_PICT", "year", "population"]]
)

# SPC's own labels, with one substitution: the codelist gives Nauru its
# endonym "Naoero", which will not match the name every other file in
# data_viz/ uses.
names = fetch_structure_pacific("DF_POP_PROJ")["dimensions"]["GEO_PICT"]
names["NR"] = "Nauru"
iso3 = {c: pycountry.countries.get(alpha_2=c).alpha_3 for c in gen["GEO_PICT"].unique()}

pacific = (
    gen.merge(pop, on=["GEO_PICT", "year"])
    .assign(
        iso_code=lambda d: d["GEO_PICT"].map(iso3),
        name=lambda d: d["GEO_PICT"].map(names),
        per_capita_electricity=lambda d: d["total_gwh"] * 1e6 / d["population"],
    )
)

print(pacific[pacific["year"] == YEAR_SNAP]
      [["name", "total_gwh", "oil_share_elec", "renewables_share_elec",
        "per_capita_electricity"]]
      .sort_values("oil_share_elec", ascending=False)
      .to_string(index=False, float_format=lambda v: f"{v:,.1f}"))

### Cross-check against the World Bank

`EG.ELC.FOSL.ZS` is the World Bank's fossil share of electricity output, built
independently of SPC. If the two agree on the islands they share, the Pacific
numbers above can be trusted next to the OWID world figures in section 2.

In [ ]:
def world_bank(indicator, date):
    url = (f"https://api.worldbank.org/v2/country/all/indicator/{indicator}"
           f"?format=json&per_page=20000&date={date}")
    try:
        payload = requests.get(url, timeout=60).json()
    except Exception as exc:
        # python.org builds on macOS ship without CA certificates until you run
        # /Applications/Python\ 3.x/Install\ Certificates.command
        print(f"Direct read failed ({type(exc).__name__}); retrying without TLS verification.")
        import urllib3

        urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
        payload = requests.get(url, timeout=60, verify=False).json()

    return pd.DataFrame([
        {"iso_code": r["countryiso3code"], "year": int(r["date"]), "value": r["value"]}
        for r in payload[1]
        if r["value"] is not None and len(r["countryiso3code"]) == 3
    ])


wb_fossil = world_bank("EG.ELC.FOSL.ZS", f"{min(YEARS)}:{YEAR_SNAP}")

check = pacific.merge(
    wb_fossil.rename(columns={"value": "wb_fossil_share"}), on=["iso_code", "year"]
)
check["gap"] = (check["fossil_share_elec"] - check["wb_fossil_share"]).abs()

print(f"{len(check)} island-years in both; median gap "
      f"{check['gap'].median():.2f} points, 90th percentile {check['gap'].quantile(0.9):.2f}")
print(check[check["year"] == 2019]
      [["name", "fossil_share_elec", "wb_fossil_share"]]
      .sort_values("fossil_share_elec", ascending=False)
      .to_string(index=False, float_format=lambda v: f"{v:,.1f}"))

## 2. World — Our World in Data

OWID's energy dataset carries the same breakdown for ~210 countries. It has no
rows at all for Palau, the Marshall Islands, Micronesia or Tuvalu, which is why
the Pacific comes from SPC and only the rest of the world comes from here.
American Samoa is the reverse case — no `DF_POWER_GEN` rows, so OWID supplies it.

In [ ]:
OWID_ENERGY = "https://github.com/owid/energy-data/raw/master/owid-energy-data.csv"

try:
    energy = pd.read_csv(OWID_ENERGY)
except Exception as exc:
    print(f"Direct read failed ({type(exc).__name__}); retrying without TLS verification.")
    import io
    import urllib3

    urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
    energy = pd.read_csv(io.StringIO(requests.get(OWID_ENERGY, verify=False).text))

world = (
    energy[
        (energy["year"] == YEAR_SNAP)
        & energy["iso_code"].notna()
        & (energy["iso_code"].str.len() == 3)
        & energy["oil_share_elec"].notna()
    ]
    [["iso_code", "country", "oil_share_elec", "fossil_share_elec",
      "renewables_share_elec", "per_capita_electricity"]]
    .rename(columns={"country": "name"})
)

print(f"{len(world)} countries for {YEAR_SNAP}")
print(f"missing from OWID: "
      f"{sorted(set(pacific['iso_code']) - set(energy['iso_code'].dropna()))}")

## 3. One ranking

Pacific islands take their figures from SPC, every other country from OWID.

In [ ]:
snapshot_pac = (
    pacific[pacific["year"] == YEAR_SNAP]
    [["iso_code", "name", "oil_share_elec", "fossil_share_elec",
      "renewables_share_elec", "per_capita_electricity"]]
    .assign(source="PDH DF_POWER_GEN", is_pacific=True)
)

ranking = (
    pd.concat([
        world[~world["iso_code"].isin(snapshot_pac["iso_code"])]
        .assign(source="OWID energy", is_pacific=False),
        snapshot_pac,
    ], ignore_index=True)
    .sort_values("oil_share_elec", ascending=False)
    .reset_index(drop=True)
)
ranking["rank_oil"] = ranking.index + 1

medians = {
    "oil_share_elec": ranking["oil_share_elec"].median(),
    "renewables_share_elec": ranking["renewables_share_elec"].median(),
    "per_capita_electricity": ranking["per_capita_electricity"].median(),
}

print(f"{len(ranking)} countries, {ranking['is_pacific'].sum()} of them Pacific")
print(f"world median oil share {medians['oil_share_elec']:.1f}%, "
      f"median {medians['per_capita_electricity']:,.0f} kWh per person")
print()
print(ranking[ranking["is_pacific"]]
      [["rank_oil", "name", "oil_share_elec", "renewables_share_elec",
        "per_capita_electricity"]]
      .to_string(index=False, float_format=lambda v: f"{v:,.1f}"))

## 4. The broader measure — SDG 7.2.1

Electricity is a slice. `EG_FEC_RNEW` is the renewable share of *total final
energy consumption*, so it carries transport and cooking too — the sectors that
dominate Palau's emissions. The World Bank publishes the same indicator for the
rest of the world, but its coverage collapses after 2020 (71 countries in 2022),
so the comparable year is 2020 even though SPC runs to 2022.

In [ ]:
YEAR_TFEC = 2020

sdg = fetch_data_pacific(
    source="DF_SDG",
    start_period=START,
    end_period="2022",
    key="A.EG_FEC_RNEW.._T._T._T._T._T._T._Z._T",
    v="3.0",
)
sdg = (
    sdg[~sdg["GEO_PICT"].isin(AGGREGATES)]
    .assign(
        year=lambda d: d["TIME_PERIOD"].astype(int),
        iso_code=lambda d: d["GEO_PICT"].map(
            lambda c: pycountry.countries.get(alpha_2=c).alpha_3
        ),
        name=lambda d: d["GEO_PICT"].map(names),
    )
    .rename(columns={"value": "renewable_share_tfec"})
    [["iso_code", "name", "year", "renewable_share_tfec"]]
)

wb_tfec = world_bank("EG.FEC.RNEW.ZS", f"{YEAR_TFEC}:{YEAR_TFEC}")
world_median_tfec = wb_tfec["value"].median()

print(f"PDH: {sdg['iso_code'].nunique()} islands, {sdg['year'].min()}-{sdg['year'].max()}")
print(f"World Bank {YEAR_TFEC}: {len(wb_tfec)} countries, median "
      f"{world_median_tfec:.1f}%")
print()
print(sdg[sdg["year"] == YEAR_TFEC].nsmallest(6, "renewable_share_tfec")
      .to_string(index=False, float_format=lambda v: f"{v:,.2f}"))

## 5. The two beats

Palau is the argument. Tokelau is the counter-argument — the one island in the
table that got off diesel, and proof the rest is a choice rather than a fate.

In [ ]:
for iso in ["PLW", "TKL"]:
    s = pacific[pacific["iso_code"] == iso]
    print(f"--- {s['name'].iloc[0]}")
    print(s[s["year"] >= 2018]
          [["year", "total_gwh", "OIL", "SOLPV", "oil_share_elec",
            "per_capita_electricity"]]
          .to_string(index=False, float_format=lambda v: f"{v:,.1f}"))
    print()

palau = ranking[ranking["iso_code"] == "PLW"].iloc[0]
print(f"Palau: {palau['oil_share_elec']:.1f}% oil against a world median of "
      f"{medians['oil_share_elec']:.1f}%, and "
      f"{palau['per_capita_electricity']:,.0f} kWh per resident against "
      f"{medians['per_capita_electricity']:,.0f} — "
      f"{palau['per_capita_electricity'] / medians['per_capita_electricity']:.1f}x "
      f"the world's electricity per person, almost all of it burned oil.")

## 6. Write outputs

In [ ]:
VIZ.mkdir(exist_ok=True)

# One long table: the Pacific with its full history, every other country at the
# snapshot year only. `is_pacific` and `year` separate them.
pacific_out = (
    pacific.assign(
        oil_gwh=lambda d: d["OIL"],
        is_pacific=True,
        source="PDH DF_POWER_GEN",
    )
    .merge(sdg[["iso_code", "year", "renewable_share_tfec"]],
           on=["iso_code", "year"], how="left")
    [["iso_code", "name", "year", "population", "total_gwh", "oil_gwh",
      "fossil_share_elec", "oil_share_elec", "renewables_share_elec",
      "per_capita_electricity", "renewable_share_tfec", "is_pacific", "source"]]
)

world_out = (
    ranking[~ranking["is_pacific"]]
    .assign(year=YEAR_SNAP, population=pd.NA, total_gwh=pd.NA, oil_gwh=pd.NA)
    .merge(wb_tfec.rename(columns={"value": "renewable_share_tfec"})
           .drop(columns="year"), on="iso_code", how="left")
    [["iso_code", "name", "year", "population", "total_gwh", "oil_gwh",
      "fossil_share_elec", "oil_share_elec", "renewables_share_elec",
      "per_capita_electricity", "renewable_share_tfec", "is_pacific", "source"]]
)

out = pd.concat([pacific_out, world_out], ignore_index=True)
out.to_csv(VIZ / "energy.csv", index=False)

payload = {
    "meta": {
        "measure": "oil-fired share of electricity generation, and electricity per resident",
        "wording": (
            "every source classes petroleum products as 'oil'; Pacific power "
            "stations run on diesel but no dataset labels it that way — write "
            "'oil-fired'"
        ),
        "snapshot_year": YEAR_SNAP,
        "tfec_year": YEAR_TFEC,
        "countries": int(len(ranking)),
        "pacific_islands": int(ranking["is_pacific"].sum()),
        "world_medians": {k: round(float(v), 2) for k, v in medians.items()},
        "world_median_renewable_share_tfec": round(float(world_median_tfec), 2),
        "sources": (
            "Pacific Data Hub .Stat (SPC) DF_POWER_GEN, DF_POP_PROJ, DF_SDG "
            "(EG_FEC_RNEW); Our World in Data energy dataset; World Bank "
            "EG.ELC.FOSL.ZS and EG.FEC.RNEW.ZS"
        ),
    },
    "snapshot": json.loads(
        ranking[["iso_code", "name", "rank_oil", "oil_share_elec",
                 "fossil_share_elec", "renewables_share_elec",
                 "per_capita_electricity", "is_pacific"]]
        .to_json(orient="records")
    ),
    "pacific_series": json.loads(
        pacific_out[["iso_code", "year", "population", "total_gwh",
                     "oil_share_elec", "renewables_share_elec",
                     "per_capita_electricity", "renewable_share_tfec"]]
        .to_json(orient="records")
    ),
}

(VIZ / "energy.json").write_text(json.dumps(payload))

print(f"{len(out)} rows -> data_viz/energy.csv")
print(f"{len(payload['snapshot'])} countries, "
      f"{len(payload['pacific_series'])} Pacific island-years -> data_viz/energy.json")

## 7. Caveats

- **"Oil" is a class, not diesel.** Pacific power stations burn diesel, but the
  data says petroleum products. Describe, don't relabel.
- **Two sources in one ranking.** Pacific islands are SPC, everyone else is
  OWID. The World Bank cross-check in section 1 is what justifies putting them
  on the same axis; it is not a guarantee of identical method.
- **American Samoa comes from OWID**, not SPC — `DF_POWER_GEN` has no rows for
  it. Palau, the Marshall Islands, Micronesia and Tuvalu are the reverse.
- **The world snapshot is one year.** `energy.csv` carries 2000–2023 for the
  Pacific but only the snapshot year for everyone else; OWID's coverage is
  thinner in older years and the ranking would shift for reasons that are not
  about energy.
- **The SDG comparison is pinned to 2020.** SPC runs to 2022, but the World
  Bank's world coverage drops to 71 countries in 2022, so a later year would
  compare the Pacific against a shrinking, non-random set.
- **Generation, not consumption.** Distribution losses and imported electricity
  are not accounted for. For islands with no interconnectors that is close to
  the same thing; for New Caledonia and PNG less so.